# Μάθημα 18: Ασφάλεια Πρακτόρων Τεχνητής Νοημοσύνης με Κρυπτογραφικές Αποδείξεις

## Πρακτικό Τετράδιο

Αυτό το τετράδιο περιλαμβάνει τέσσερις ασκήσεις:

1. **Υπογράψτε την πρώτη σας απόδειξη** για μια κλήση εργαλείου πράκτορα και επαληθεύστε την.
2. **Παραποιήστε την απόδειξη** και δείτε την επαλήθευση να αποτυγχάνει.
3. **Δημιουργήστε μια αλυσίδα τριών αποδείξεων** και επιβεβαιώστε την ακεραιότητα της αλυσίδας.
4. **Ενσωματώστε μια κλήση εργαλείου Microsoft Agent Framework** ώστε κάθε ενέργεια να παράγει μια απόδειξη.

Όλα τα κρυπτογραφικά πρωτόκολλα εισάγονται από αξιόπιστες βιβλιοθήκες (`pynacl` για Ed25519, `jcs` για το RFC 8785 canonical JSON, `hashlib` από τη βασική βιβλιοθήκη Python για SHA-256). Η λογική της απόδειξης είναι απλός Python κώδικας που μπορείτε να διαβάσετε και να τροποποιήσετε.

Εκτελέστε τα κελιά με τη σειρά. Κάθε ενότητα είναι σύντομη και αυτοτελής.


## Εγκατάσταση

Εγκαταστήστε τις δύο εξαρτήσεις. Και οι δύο έχουν άδειες χρήσης με άδεια (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Βοηθητικά Εργαλεία

Αυτά τα δύο βοηθητικά χειρίζονται την κωδικοποίηση base64url (χωρίς συμπλήρωση) και την κατακερματισμό SHA-256 αυθαίρετων αντικειμένων. Κρατούν το υπόλοιπο τετράδιο εστιασμένο στην ίδια τη λογική της απόδειξης.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Ενότητα 1: Υπογράψτε την πρώτη σας απόδειξη

Φανταστείτε ότι ο εκπρόσωπός μας για την **Contoso Travel** μόλις αναζήτησε πτήσεις από το Σίδνεϊ προς το Λος Άντζελες για έναν πελάτη. Θέλουμε να καταγράψουμε τη δήλωση της πύλης σχετικά με αυτήν την κλήση εργαλείου ώστε ένας μελλοντικός ελεγκτής να μπορεί να την επαληθεύσει με ένα ξεχωριστά καρφιτσωμένο κλειδί εκδότη.

### Βήμα 1.1: Δημιουργία κλειδιού υπογραφής

Σε παραγωγικό περιβάλλον, το κλειδί υπογραφής του αντιπροσώπου θα βρισκόταν σε έναν εξοπλισμό ασφαλείας υλικού (HSM), το Azure Key Vault, ή μια παρόμοια προστατευμένη αποθήκη. Για αυτό το μάθημα δημιουργούμε ένα φρέσκο κλειδί στη μνήμη.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Βήμα 1.2: Δημιουργία του περιεχομένου απόδειξης

Το περιεχόμενο περιέχει όλα όσα θέλουμε να βεβαιώνει η απόδειξη: ποιος ενήργησε, ποιο εργαλείο, με ποια επιχειρήματα, τι επέστρεψε, υπό ποια πολιτική και πότε. Αποθηκεύουμε με κατακερματισμό τα επιχειρήματα και το αποτέλεσμα αντί να τα συμπεριλάβουμε απευθείας, ώστε η απόδειξη να μην αποκαλύπτει ευαίσθητο περιεχόμενο.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Βήμα 1.3: Υπογράψτε και συναρμολογήστε την απόδειξη

Τρία βήματα:

1. Κανινικοποιήστε το περιεχόμενο χρησιμοποιώντας JCS ώστε δύο υλοποιήσεις που παράγουν την ίδια λογική απόδειξη να παράγουν byte-ταυτόσημα bytes.
2. Υπογράψτε τα κανονικοποιημένα bytes απευθείας με το ιδιωτικό κλειδί Ed25519. Το PureEdDSA κατακερματίζει το μήνυμα εσωτερικά, οπότε ένα επιπλέον προκατακερματισμό θα άλλαζε το πρωτόκολλο.

Στη συνέχεια, η υπογραφή επισυνάπτεται στο αρχικό περιεχόμενο για να παραχθεί η τελική απόδειξη.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Βήμα 1.4: Επαλήθευση της απόδειξης

Η επαλήθευση αναιρεί τη διαδικασία. Αφαιρούμε την υπογραφή, επανυπολογίζουμε τα κανονικά bytes και ελέγχουμε την υπογραφή σε σχέση με το δημόσιο κλειδί του εκδότη.

Η επαλήθευση είναι εκτός σύνδεσης: δεν απαιτείται κλήση σε υπηρεσία και δεν υπάρχει πρόσβαση σε δίκτυο. Απαιτεί όμως ένα πράγμα εκτός από την απόδειξη: το δημόσιο κλειδί του εκδότη, που λαμβάνεται μέσω καναλιού που ήδη εμπιστεύεται ο επόπτης, όπως ένα JWK Set που δημοσιεύει ο εκδότης ή μια τιμή που έχει καρφιτσωθεί εκ των προτέρων. Το πεδίο `public_key` μέσα στην απόδειξη δεν μπορεί να παίξει αυτόν τον ρόλο, επειδή όποιος γράφει την απόδειξη γράφει και αυτό το πεδίο. Το Τμήμα 2 δείχνει τη νοθεία που αυτό αποτρέπει.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Πρέπει να δείτε `Receipt is valid: True`, `Pre-hashed receipt valid: False` και `Malformed receipts rejected: True`. Η θετική περίπτωση αποδεικνύει ότι η άμεση διαδρομή υπογραφής JCS λειτουργεί· ο αρνητικός έλεγχος κάνει τον κανόνα εύρους υπογραφής εκτελέσιμο αντί να τον αφήνει ως πρόζα· η τελευταία γραμμή δείχνει κακοσχηματισμένη είσοδο που επιστρέφει `False` αντί να προκαλεί εξαίρεση.


## Ενότητα 2: Παραποίηση της απόδειξης

Ολόκληρος ο σκοπός των αποδείξεων είναι ότι είναι ανιχνεύσιμες στην παραποίηση. Ας το αποδείξουμε.

Θα τροποποιήσουμε ακριβώς ένα χαρακτήρα της απόδειξης και θα παρατηρήσουμε την αποτυχία επαλήθευσης.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Τι μόλις συνέβη;

Όταν αλλάξαμε το `policy_id`, τα κανονικά byte άλλαξαν. Η υπογραφή (η οποία ήταν πάνω από τα αρχικά κανονικά byte) δεν ταιριάζει πια. Η επαλήθευση επιστρέφει σωστά `False`.

Δεν υπάρχει τρόπος να αλλάξεις ένα υπογεγραμμένο πεδίο, δηλαδή οτιδήποτε εκτός από το αντικείμενο `signature`, και να έχει η απόδειξη έγκυρη επαλήθευση με το δημόσιο κλειδί του εκδότη, εκτός αν ο επιτιθέμενος έχει το ιδιωτικό κλειδί του εκδότη. Το ίδιο το αντικείμενο `signature` δεν είναι υπογεγραμμένο: `sig` είναι η υπογραφή, και `verify_receipt` αγνοεί το `public_key`, οπότε η επεξεργασία μόνο αυτού του πεδίου δεν αλλάζει τίποτα. Εφόσον το ιδιωτικό κλειδί βρίσκεται σε θησαυροφυλάκιο κλειδιών και οι επαληθευτές χρησιμοποιούν το ανεξάρτητα δημοσιευμένο κλειδί του εκδότη, είναι αδύνατο να κρυφτεί μια αλλαγή σε υπογεγραμμένο πεδίο. Αυτή η προϋπόθεση έχει σημασία: το επόμενο κελί δείχνει μια τροποποιημένη απόδειξη που επαληθεύεται άψογα με διαφορετικό κλειδί.

Δοκιμάστε το μόνοι σας: τροποποιήστε το `tool_name` ή `agent_id` ή `timestamp` στο παραπάνω κελί και ξανατρέξτε. Κάθε αλλαγή σε υπογεγραμμένο πεδίο παράγει μια άκυρη απόδειξη.


### Η παραποίηση που η υπογραφή δεν ανιχνεύει

Ο παραπάνω επιτιθέμενος δεν υπέγραψε ξανά. Κάποιος πιο προσεκτικός θα: τροποποιούσε την απόδειξη,
θα την υπέγραφε με κλειδί που δημιούργησε ο ίδιος, και θα έβαζε αυτό το κλειδί στο
`public_key` πεδίο. Το αποτέλεσμα είναι μια απόδειξη της οποίας η υπογραφή είναι απολύτως γνήσια
για το περιεχόμενό της.

Τρέξε το αυτό και δες ποιος από τους δύο ελέγχους το ανιχνεύει.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Πρέπει να δείτε `True` και μετά `False`. Οι τρεις γραμμές `assert` κάνουν αυτό έναν
έλεγχο: το κελί προκαλεί σφάλμα αν η παραποίηση ποτέ επιβεβαιωθεί έναντι του
κλειδιού του εκδότη.

Η παραποίηση είναι εσωτερικά συνεπής. Η υπογραφή της όντως ταιριάζει με τα
περιεχόμενά της και το κλειδί που φέρει, οπότε ένας επαληθευτής που διαβάζει το κλειδί από την
απόδειξη την αναφέρει ως έγκυρη. Τίποτα σχετικά με την απόδειξη δεν φαίνεται λάθος, επειδή
τίποτα σε αυτή δεν είναι ασυνεπές.

Το μόνο που τη συλλαμβάνει είναι ο έλεγχος έναντι ενός κλειδιού που ο επιτιθέμενος δεν είχε
τρόπο να επηρεάσει. Γι’ αυτό το `verify_receipt` παίρνει το κλειδί ως όρισμα: μια
συνάρτηση με την υπογραφή `verify_receipt(receipt)` δεν μπορεί να γραφτεί με ασφάλεια,
γιατί το μόνο κλειδί που θα μπορούσε να φτάσει είναι αυτό που παρείχε ο επιτιθέμενος.

Η ίδια απόδειξη αποθηκεύεται ως `sample_receipts/04_resigned_receipt.json` αν
θέλετε να την επιθεωρήσετε έξω από το σημειωματάριο.


## Ενότητα 3: Συνδέστε τις αποδείξεις μεταξύ τους

Μια μεμονωμένη απόδειξη προστατεύει μια ενέργεια. Οι περισσότεροι παράγοντες εκτελούν πολλές ενέργειες. Για να γίνει ολόκληρη η ακολουθία ανιχνεύσιμη ως προς παραποιήσεις, συνδέουμε κάθε απόδειξη με την προηγούμενη, συμπεριλαμβάνοντας το hash της προηγούμενης απόδειξης στο payload της νέας απόδειξης.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Εάν κάποιος αφαιρέσει ή αναδιατάξει μια απόδειξη μέσα στην παρεχόμενη ακολουθία, ο επόμενος σύνδεσμος σπάει. Η ανίχνευση αποκοπής ή επανεγγραφής επισημείου υπό έλεγχο υπογράφοντα απαιτεί επίσης ένα καρφιτσωμένο τρέχον κεφάλι, εξωτερικό σημείο ελέγχου ή αυθεντικοποιημένη δέσμευση πληθυσμού.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Τώρα σπάστε την αλυσίδα παραποιώντας την μέση απόδειξη και επαληθεύστε ξανά. Η παραποιημένη απόδειξη αποτυγχάνει στον έλεγχο της υπογραφής της, ΚΑΙ η επόμενη απόδειξη αποτυγχάνει στον έλεγχο της αλυσίδας (επειδή το `previous_receipt_hash` της δεν ταιριάζει πλέον με το τροποποιημένο hash της μέσης απόδειξης).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Η απόδειξη 0 εξακολουθεί να επαληθεύεται (δεν τροποποιήθηκε και δεν έχει προκατόχο από τον οποίο να εξαρτάται). Η απόδειξη 1 αποτυγχάνει τον έλεγχο υπογραφής της επειδή αλλάξαμε το `tool_args_hash`. Η απόδειξη 2 αποτυγχάνει τον έλεγχο αλυσίδας της επειδή το `previous_receipt_hash` της υπολογίστηκε σε σχέση με την αρχική (τώρα τροποποιημένη) απόδειξη 1.

Ακόμα και αν ένας επιτιθέμενος επανυπογράψει την τροποποιημένη απόδειξη 1, η νέα υπογραφή αποτυγχάνει με το κλειδί του εκδότη εκτός αν κατέχει το ιδιωτικό κλειδί του εκδότη (μια επανυπογραφή με το δικό τους κλειδί αποτυγχάνει, όπως δείχνει η Ενότητα 2), και η ασυμφωνία στην αλυσίδα στην απόδειξη 2 θα εξακολουθούσε να αποκαλύπτει την παραποίηση. Για να κρύψει την αλλαγή, ο επιτιθέμενος θα έπρεπε να επανυπογράψει κάθε απόδειξη από το σημείο της τροποποίησης και μετά, κάτι που απαιτεί κατοχή του ιδιωτικού κλειδιού του εκδότη.


## Ενότητα 4: Περιτύλιγμα μιας κλήσης εργαλείου αντιπροσώπου με υπογραφή παραλαβής

Σε μια πραγματική ανάπτυξη, δεν θέλετε κάθε δημιουργό αντιπροσώπου να θυμάται να καλεί `make_receipt`. Θέλετε η υπογραφή παραλαβής να είναι αυτόματη για κάθε κλήση εργαλείου.

Εδώ είναι το απλούστερο μοτίβο: μια κλάση περιτύλιξης που παίρνει οποιαδήποτε κλητή συνάρτηση εργαλείου και επιστρέφει μια έκδοση που εκπέμπει παραλαβή. Αυτό προσαρμόζεται σε οποιοδήποτε πλαίσιο αντιπροσώπου, συμπεριλαμβανομένου του Microsoft Agent Framework (`agent_framework.foundry`).

Αν δεν έχετε ρυθμίσει ένα έργο Microsoft Foundry, το τοπικό mock παρακάτω δείχνει ακόμα το μοτίβο.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Ενσωμάτωση με το Microsoft Agent Framework

Ο `ReceiptedTool` περιτυλιγμένος παραπάνω είναι ανεξάρτητος πλαισίου. Για να τον χρησιμοποιήσετε μέσα σε έναν πράκτορα που έχει κατασκευαστεί με το Microsoft Agent Framework, καταχωρήστε τη συνάρτηση που έχει τυλιχτεί ως εργαλείο. Ένα σχέδιο (θα αντικαταστήσετε το ψεύτικο με μια πραγματική καταχώρηση εργαλείου Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Το πλαίσιο πράκτορα δεν χρειάζεται να γνωρίζει τίποτα για τις αποδείξεις. Η υπογραφή απόδειξης είναι περιτυλιγμένη γύρω από το εργαλείο, όχι ενσωματωμένη στο πλαίσιο. Έτσι προσθέτετε πηγή στα υπάρχοντα κώδικα πράκτορα χωρίς να ξαναγράψετε τον πράκτορα.


## Ανακεφαλαίωση και πρόκληση εκτάσης

Έχετε:

- Δημιουργήσει ένα ζεύγος κλειδιών Ed25519.
- Δημιουργήσει και υπογράψει μια απόδειξη για μια κλήση εργαλείου πράκτορα.
- Επαληθεύσει την απόδειξη εκτός σύνδεσης χρησιμοποιώντας μόνο το δημόσιο κλειδί του εκδότη, που αποκτήθηκε ανεξάρτητα από την απόδειξη.
- Παρέμβει σε μια απόδειξη και παρατήρησε αποτυχία επαλήθευσης.
- Δημιουργήσει μια αλληλουχία τριών αποδείξεων με κατακερματισμένη αλυσίδα.
- Παρέμβει στο μέσο της αλυσίδας και παρατήρησε αποτυχία τόσο της υπογραφής όσο και του συνδέσμου της αλυσίδας.
- Περιέκλεισε μια λειτουργία εργαλείου με αυτόματη υπογραφή απόδειξης.

**Πρόκληση επέκτασης.** Επεκτείνετε το σχήμα της απόδειξης με ένα `request_id` πεδίο (ένα UUID για κατανεμημένη ιχνηλάτηση). Ενημερώστε το `make_receipt` ώστε να το περιλαμβάνει, και επιβεβαιώστε ότι οι αποδείξεις εξακολουθούν να επαληθεύονται από άκρο σε άκρο. Στη συνέχεια, τροποποιήστε το πεδίο μετά την υπογραφή και επιβεβαιώστε ότι η επαλήθευση αποτυγχάνει. Αυτό σας αναγκάζει να εσωτερικεύσετε πώς κάθε byte της κανονικής κωδικοποίησης συμβάλλει στην υπογραφή.

**Σημαντικό όριο.** Υπό μια σταθερή κλειδί και πλαίσιο αλυσίδας, οι αποδείξεις αποδεικνύουν την ανάθεση υπογραφής, την ακεραιότητα περιεχομένου και έναν σύνδεσμο με τον παρουσιαζόμενο προκάτοχο. ΔΕΝ αποδεικνύουν εκτέλεση, πηγή αλήθειας, αξιόπιστο χρόνο, πληρότητα, αξιολόγηση πολιτικής ή ορθότητα. Οι αποδείξεις είναι ένα θεμέλιο. Η διακυβέρνηση είναι το σύστημα που χτίζετε πάνω.

Διαβάστε ξανά το README του μαθήματος με το εν λόγω όριο κατά νου. Το πιο κοινό λάθος που κάνουν οι ομάδες με τις αποδείξεις είναι να υποθέτουν ότι "έχουμε αποδείξεις" σημαίνει "είμαστε διακυβερνημένοι." Δεν σημαίνει. Οι αποδείξεις καθιστούν τη συμπεριφορά του πράκτορα ελέγξιμη. Δεν την καθιστούν σωστή.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
